# Uber Assignment NB Solution

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2023, University of Chicago 

---

In [17]:
from pprint import pprint


In [18]:
drivers = ['D32', 'D8712', 'D922', 'D88']
customers = ['C11', 'C934', 'C331']


In [19]:
travel_times = {('D32', 'C11'): 587,
 ('D32', 'C331'): 1380,
 ('D32', 'C934'): 915,
 ('D8712', 'C11'): 1734,
 ('D8712', 'C331'): 693,
 ('D8712', 'C934'): 974,
 ('D88', 'C11'): 621,
 ('D88', 'C331'): 1230,
 ('D88', 'C934'): 830,
 ('D922', 'C11'): 1462,
 ('D922', 'C331'): 472,
 ('D922', 'C934'): 958}

In [20]:
from pulp import *
model = LpProblem("Uber_Assignment",LpMinimize)

In [21]:
X= {} 
for driver in drivers: 
    for customer in customers:
        X[(driver,customer)] = LpVariable(f"X_{driver}_{customer}", cat='Binary')
print(f'Flow Variables X: {X}')

Flow Variables X: {('D32', 'C11'): X_D32_C11, ('D32', 'C934'): X_D32_C934, ('D32', 'C331'): X_D32_C331, ('D8712', 'C11'): X_D8712_C11, ('D8712', 'C934'): X_D8712_C934, ('D8712', 'C331'): X_D8712_C331, ('D922', 'C11'): X_D922_C11, ('D922', 'C934'): X_D922_C934, ('D922', 'C331'): X_D922_C331, ('D88', 'C11'): X_D88_C11, ('D88', 'C934'): X_D88_C934, ('D88', 'C331'): X_D88_C331}


In [22]:
# Let's loop through OUR variable names, like 'C1_D2', 
# from those we can access the distance and true PuLP variable objects
obj = ''
for driver in drivers: 
    for customer in customers: 
        obj += travel_times[(driver,customer)] * X[(driver,customer)]
model += obj, "Cost of Customer Driver Assignment"
print(model)

Uber_Assignment:
MINIMIZE
587*X_D32_C11 + 1380*X_D32_C331 + 915*X_D32_C934 + 1734*X_D8712_C11 + 693*X_D8712_C331 + 974*X_D8712_C934 + 621*X_D88_C11 + 1230*X_D88_C331 + 830*X_D88_C934 + 1462*X_D922_C11 + 472*X_D922_C331 + 958*X_D922_C934 + 0
VARIABLES
0 <= X_D32_C11 <= 1 Integer
0 <= X_D32_C331 <= 1 Integer
0 <= X_D32_C934 <= 1 Integer
0 <= X_D8712_C11 <= 1 Integer
0 <= X_D8712_C331 <= 1 Integer
0 <= X_D8712_C934 <= 1 Integer
0 <= X_D88_C11 <= 1 Integer
0 <= X_D88_C331 <= 1 Integer
0 <= X_D88_C934 <= 1 Integer
0 <= X_D922_C11 <= 1 Integer
0 <= X_D922_C331 <= 1 Integer
0 <= X_D922_C934 <= 1 Integer



In [23]:
#  Or, of course, let's use Python loops!!
# First the driver constraints
for driver in drivers:
    const = ''
    for customer in customers:
        const += X[(driver,customer)]
    model += const <= 1, f"driver_{driver}"
# Next let's do the customer constraints
for customer in customers:
    const = ''
    for driver in drivers:
        const += X[(driver,customer)]
    model += const <= 1, f"customer_{customer}"


In [24]:
# Ensure that the number of total assignments equals min(#drivers, #customers)
num_assignments = min(len(drivers),len(customers))
print("Number of total assignments:",num_assignments)


Number of total assignments: 3


In [25]:
const = None 
for driver in drivers:
    for customer in customers:
        const += X[(driver,customer)]
model += const == num_assignments, f"Number_of_Assignments"

In [26]:
print(model)

Uber_Assignment:
MINIMIZE
587*X_D32_C11 + 1380*X_D32_C331 + 915*X_D32_C934 + 1734*X_D8712_C11 + 693*X_D8712_C331 + 974*X_D8712_C934 + 621*X_D88_C11 + 1230*X_D88_C331 + 830*X_D88_C934 + 1462*X_D922_C11 + 472*X_D922_C331 + 958*X_D922_C934 + 0
SUBJECT TO
driver_D32: X_D32_C11 + X_D32_C331 + X_D32_C934 <= 1

driver_D8712: X_D8712_C11 + X_D8712_C331 + X_D8712_C934 <= 1

driver_D922: X_D922_C11 + X_D922_C331 + X_D922_C934 <= 1

driver_D88: X_D88_C11 + X_D88_C331 + X_D88_C934 <= 1

customer_C11: X_D32_C11 + X_D8712_C11 + X_D88_C11 + X_D922_C11 <= 1

customer_C934: X_D32_C934 + X_D8712_C934 + X_D88_C934 + X_D922_C934 <= 1

customer_C331: X_D32_C331 + X_D8712_C331 + X_D88_C331 + X_D922_C331 <= 1

Number_of_Assignments: X_D32_C11 + X_D32_C331 + X_D32_C934 + X_D8712_C11
 + X_D8712_C331 + X_D8712_C934 + X_D88_C11 + X_D88_C331 + X_D88_C934
 + X_D922_C11 + X_D922_C331 + X_D922_C934 = 3

VARIABLES
0 <= X_D32_C11 <= 1 Integer
0 <= X_D32_C331 <= 1 Integer
0 <= X_D32_C934 <= 1 Integer
0 <= X_D8712_C11 <

In [27]:
# Let's solve the model and make sure it's status is good
model.solve()
print("Status:", LpStatus[model.status])

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/dde/opt/anaconda3/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/64/cbc /var/folders/cb/dn9jvkgx2z5f15qppmk24pk00000gn/T/b60f0d1e5fd14083a945a58d40c07d87-pulp.mps timeMode elapsed branch printingOptions all solution /var/folders/cb/dn9jvkgx2z5f15qppmk24pk00000gn/T/b60f0d1e5fd14083a945a58d40c07d87-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 13 COLUMNS
At line 86 RHS
At line 95 BOUNDS
At line 108 ENDATA
Problem MODEL has 8 rows, 12 columns and 36 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1889 - 0.00 seconds
Cgl0004I processed model has 8 rows, 12 columns (12 integer (12 of which binary)) and 36 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of 1889
Cbc0038I Before mini branch

In [28]:
# Here is the solution
for v in model.variables():
    print(v.name, "=", v.varValue)

X_D32_C11 = 1.0
X_D32_C331 = 0.0
X_D32_C934 = 0.0
X_D8712_C11 = 0.0
X_D8712_C331 = 0.0
X_D8712_C934 = 0.0
X_D88_C11 = 0.0
X_D88_C331 = 0.0
X_D88_C934 = 1.0
X_D922_C11 = 0.0
X_D922_C331 = 1.0
X_D922_C934 = 0.0


In [29]:
print("Total Objective Function Value is = ", value(model.objective))

Total Objective Function Value is =  1889.0
